# Модуль 05 · Урок 2. Resample, shift, diff, pct_change

**Как проходить:** изучите все разделы теории последовательно. Запустите демонстрационные ячейки, осознайте входы и выходы методов. Только после этого решайте задания. В условиях используются другие предметные области и комбинации действий — готовых решений в теории нет.

**Обещание урока:** все необходимые операции объяснены ниже; внешний поиск не нужен.

**Стандарт проверки:** правильные значения, форма, порядок и неизменность входа; минимум два набора данных на задачу.

In [ ]:
# Настройка путей для работы из любой папки модуля в VS Code
from pathlib import Path
import os
import sys

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "tests" / "notebook_checks.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Не найден корень pandas_academy. Откройте папку курса целиком "
        "через File -> Open Folder в VS Code."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # Теперь data/ и output/ всегда находятся в корне курса

import pandas as pd
import numpy as np
from tests.notebook_checks import expected_for, check_result, show_progress, show_progress

(ROOT / "output").mkdir(exist_ok=True)
print("pandas:", pd.__version__, "| корень проекта:", ROOT)


## 1. Частота наблюдений
Временной ряд должен иметь тип datetime; обычно его удобно назначить индексом. `resample('D')` группирует по календарным суткам, `resample('MS')` по началу месяца, а `resample('h')` по часам. `resample` использует временные корзины, а не одинаковые интервалы между строками. Для сумм и средних результаты могут отличаться на пропущенных днях — решите, означает ли отсутствие наблюдений ноль или неизвестное значение.

In [ ]:
import pandas as pd
hits = pd.DataFrame({'timestamp':pd.to_datetime(['2026-01-01 09:00','2026-01-01 14:00','2026-01-03 12:00']), 'count':[3,6,7]})
print(hits.set_index('timestamp').resample('D')['count'].sum())

## 2. shift — сдвиг наблюдений
`shift(1)` сопоставляет текущей строке значение предыдущей, **не сортируя данные автоматически**. Поэтому перед сравнением с прошлым периодом обязательно отсортируйте ряд. Первый элемент после сдвига обычно NA; подстановка нуля изменит смысл относительных изменений.

In [ ]:
monthly = pd.Series([120,140,133],index=pd.period_range('2026-01',periods=3,freq='M'))
print(pd.DataFrame({'now':monthly,'prev':monthly.shift(1),'growth':monthly.pct_change()}))

## 3. diff и pct_change
`diff()` вычисляет абсолютную разницу текущего значения с предыдущим. `pct_change()` возвращает **долю изменения**, не процент: для роста 10% результат равен `0.10`, а после умножения на 100 — 10. При предыдущем нуле может появиться бесконечность, которую нужно обрабатывать осознанно.

## 4. Кумулятивные расчёты
`cumsum` рассчитывает накопительную сумму, `cummax` — максимум от начала ряда. На данных нескольких пользователей сначала сортируйте события, затем выполняйте операции внутри группы `groupby('user')`. Никогда не вычисляйте накопительный итог без указания, к какой совокупности он относится.

**Проверьте себя:** как resample отличается от обычного groupby? Что нужно выполнить до shift? Почему pct_change нужно умножить на 100 для процентов?

---
## Самостоятельная практика

**Инструкция:** выполните ячейку исходных данных, напишите обычный код в следующей ячейке и сохраните итог в `result`. Под решением показан ожидаемый вывод. Затем запустите проверку. **Не нужно писать функции.**

### m5_06 · daily sum

Для ledger суммируйте amount по каждому календарному дню от минимальной до максимальной даты; отсутствующие дни должны иметь 0. Получите Series с DatetimeIndex и именем amount.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `date`, `shop`, `amount`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m5_06: используются переменные df

df = pd.DataFrame({
    'date': ['2026-01-01', '2026-01-02', '2026-01-04', '2026-01-07', '2026-01-08'],
    'shop': ['A', 'A', 'A', 'B', 'B'],
    'amount': [10.0, 25.0, 30.0, 40.0, 15.0],
})

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m5_06 = expected_for("m5_06", df=df)


In [ ]:
# m5_06: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.Series`: 8 значений; имя серии: `amount`.

```text
date
2026-01-01    10.0
2026-01-02    25.0
2026-01-03     0.0
2026-01-04    30.0
2026-01-05     0.0
2026-01-06     0.0
Freq: D
... (показаны только первые значения)
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m5_06", result, _expected_m5_06)

### m5_07 · monthly revenue

Сгруппируйте выручку `price*qty` по календарным месяцам с индексом Period[M] (отсортированным), получите Series revenue.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m5_07: используются переменные df

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m5_07 = expected_for("m5_07", df=df)


In [ ]:
# m5_07: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.Series`: 2 значений; имя серии: `revenue`.

```text
month
2026-01    889.0
2026-02    488.0
Freq: M
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m5_07", result, _expected_m5_07)

### m5_08 · daily delta

Отсортируйте ledger по date и добавьте `delta` = amount минус предыдущий amount (первый — NA).

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `date`, `shop`, `amount`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m5_08: используются переменные df

df = pd.DataFrame({
    'date': ['2026-01-01', '2026-01-02', '2026-01-04', '2026-01-07', '2026-01-08'],
    'shop': ['A', 'A', 'A', 'B', 'B'],
    'amount': [10.0, 25.0, 30.0, 40.0, 15.0],
})

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m5_08 = expected_for("m5_08", df=df)


In [ ]:
# m5_08: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 5 строк, 4 столбца; колонки: `date`, `shop`, `amount`, `delta`.

```text
         date shop  amount  delta
0  2026-01-01    A    10.0    NaN
1  2026-01-02    A    25.0   15.0
2  2026-01-04    A    30.0    5.0
3  2026-01-07    B    40.0   10.0
4  2026-01-08    B    15.0  -25.0
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m5_08", result, _expected_m5_08)

### m5_09 · cum sales by shop

Отсортируйте ledger по shop, date и добавьте `running_total` — накопительную сумму amount внутри shop.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `date`, `shop`, `amount`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m5_09: используются переменные df

df = pd.DataFrame({
    'date': ['2026-01-01', '2026-01-02', '2026-01-04', '2026-01-07', '2026-01-08'],
    'shop': ['A', 'A', 'A', 'B', 'B'],
    'amount': [10.0, 25.0, 30.0, 40.0, 15.0],
})

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m5_09 = expected_for("m5_09", df=df)


In [ ]:
# m5_09: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 5 строк, 4 столбца; колонки: `date`, `shop`, `amount`, `running_total`.

```text
         date shop  amount  running_total
0  2026-01-01    A    10.0           10.0
1  2026-01-02    A    25.0           35.0
2  2026-01-04    A    30.0           65.0
3  2026-01-07    B    40.0           40.0
4  2026-01-08    B    15.0           55.0
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m5_09", result, _expected_m5_09)

### m5_10 · relative growth

Для числовой Series получите относительное изменение между последовательными наблюдениями через pct_change, умноженное на 100; первый элемент NA.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `values` — `pd.Series` (индекс и значения определены в ячейке примера).


**Выполнение:** используйте готовые переменные `values`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m5_10: используются переменные values

values = pd.Series([100.0, 110.0, 132.0], index=[0, 1, 2])

print("values — демонстрационные данные:")

display(values)


# Снимок правильного результата для автопроверки.
_expected_m5_10 = expected_for("m5_10", values=values)


In [ ]:
# m5_10: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: values
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.Series`: 3 значений; имя серии: `None`.

```text
0     NaN
1    10.0
2    20.0
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m5_10", result, _expected_m5_10)

---
## Итоги урока

Если все проверки пройдены, повторите своими словами: **что делает каждый метод, какие параметры важны и какая ошибка наиболее вероятна**. Перед переходом к следующему уроку попробуйте решить одно задание заново с чистой ячейки. Автотест подтверждает поведение на известных наборах, но не заменяет код-ревью.

In [ ]:
show_progress()